# Step 3: Movie Recommendation System

This notebook adds the first recommendation-system module to the IMDB project. The goal is to recommend movies that are similar in content when a user selects one movie.

This version uses **Content-Based Recommendation** with **TF-IDF + Cosine Similarity**. It does not implement deep learning, LLM recommendation, or collaborative filtering in this stage.

## 1. Recommendation Goal

The system answers a simple user-facing question:

> If a user likes this movie, what other movies have similar content?

The recommendation basis is not the movie title. Each movie is represented by a content text built from:

- genres
- overview
- keywords

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.recommender import (
    DEFAULT_KEYWORDS_PATH,
    DEFAULT_METADATA_PATH,
    build_content_recommender,
    find_movie_id_by_title,
)

OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

METADATA_PATH = PROJECT_ROOT / DEFAULT_METADATA_PATH
KEYWORDS_PATH = PROJECT_ROOT / DEFAULT_KEYWORDS_PATH

pd.set_option("display.max_columns", 30)

## 2. Data Used

This module uses two existing dataset files:

- `movies_metadata.csv`: title, genres, overview, vote average, vote count, release date
- `keywords.csv`: keyword tags for each movie

These files are appropriate for a content-based recommender because they describe what each movie is about.

In [2]:
print(f"Metadata path: {METADATA_PATH}")
print(f"Keywords path: {KEYWORDS_PATH}")
print(f"Metadata exists: {METADATA_PATH.exists()}")
print(f"Keywords exists: {KEYWORDS_PATH.exists()}")

Metadata path: C:\Users\刘子豪\Desktop\code\IMDB_analysis\data\movies_metadata.csv
Keywords path: C:\Users\刘子豪\Desktop\code\IMDB_analysis\data\keywords.csv
Metadata exists: True
Keywords exists: True


## 3. Build TF-IDF Content Recommender

For each movie, the recommender creates:

`content = genres + overview + keywords`

TF-IDF converts this content text into numeric vectors. Cosine similarity then measures how close two movie vectors are. Higher cosine similarity means the movies share more similar content signals.

In [3]:
recommender, recommender_summary = build_content_recommender(
    metadata_path=METADATA_PATH,
    keywords_path=KEYWORDS_PATH,
)

summary_table = pd.DataFrame(recommender_summary.items(), columns=["Item", "Value"])
summary_table

,Item,Value
0,metadata_rows_loaded,2802
1,keywords_rows_loaded,46419
2,recommendation_rows,2798
3,movies_with_keywords,2454
4,movies_with_overview,2784
5,movies_with_genres,2730
6,tfidf_rows,2798
7,tfidf_features,30000


## 4. Recommendation Dataset Preview

The final recommendation dataset keeps only the fields needed for content matching and user-facing output.

In [4]:
recommender.movies[[
    "movie_id", "title", "release_year", "main_genre", "genres_text", "keywords_text", "vote_average", "vote_count"
]].head()

,movie_id,title,release_year,main_genre,genres_text,keywords_text,vote_average,vote_count
0,101230,1-900,1994.0,Unknown,,,5.0,1
1,4951,10 Things I Hate About You,1999.0,Comedy,comedy romance drama,shakespeare sister high_school cannabis decept...,7.3,1768
2,11674,101 Dalmatians,1996.0,Comedy,comedy family,london_england fur fashion_designer puppy base...,5.6,722
3,389,12 Angry Men,1957.0,Drama,drama,judge jurors sultriness death_penalty father_m...,8.2,2130
4,9401,2 Days in the Valley,1996.0,Comedy,comedy crime,detective assassin jealousy hostage cocaine nu...,5.9,73


## 5. Sample Recommendation

The sample recommendation below is saved to `outputs/sample_recommendations.csv`. It is useful for quick validation and presentation.

In [5]:
sample_candidates = ["Star Wars", "The Godfather", "Batman", "Toy Story"]
sample_title = None
sample_movie_id = None

for candidate in sample_candidates:
    try:
        sample_movie_id = find_movie_id_by_title(recommender.movies, candidate)
        sample_title = candidate
        break
    except ValueError:
        continue

if sample_movie_id is None:
    sample_movie_id = int(recommender.movies.iloc[0]["movie_id"])
    sample_title = str(recommender.movies.iloc[0]["title"])

sample_recommendations = recommender.recommend_by_id(sample_movie_id, top_n=10)
sample_recommendations.insert(0, "Source Movie", sample_title)

sample_path = OUTPUT_DIR / "sample_recommendations.csv"
sample_recommendations.to_csv(sample_path, index=False)
print(f"Sample movie: {sample_title}")
print(f"Saved sample recommendations to: {sample_path}")
sample_recommendations

Sample movie: Star Wars
Saved sample recommendations to: C:\Users\刘子豪\Desktop\code\IMDB_analysis\outputs\sample_recommendations.csv


,Source Movie,Recommended Movie,Similarity Score,Vote Average,Vote Count,Main Genre
0,Star Wars,The Empire Strikes Back,0.4730,8.2,5998,Adventure
1,Star Wars,Return of the Jedi,0.2539,7.9,4763,Adventure
2,Star Wars,Star Wars: Episode I - The Phantom Menace,0.1719,6.4,4526,Adventure
3,Star Wars,Mad Dog Time,0.1504,6.2,12,Drama
4,Star Wars,The Swan Princess,0.1077,6.5,251,Animation
5,Star Wars,Sleeping Beauty,0.0859,6.8,1332,Fantasy
6,Star Wars,The Princess Bride,0.0829,7.6,1518,Adventure
7,Star Wars,Star Trek: Insurrection,0.0766,6.3,402,Science Fiction
8,Star Wars,The Mummy,0.0761,6.8,58,Horror
9,Star Wars,History of the World: Part I,0.0678,6.5,205,Comedy


## 6. Interpretation

The recommendations are based on content similarity. For example, when a franchise or genre has similar descriptions and keywords, related movies tend to receive higher similarity scores.

The output should be interpreted as: **these movies are content-similar**, not as proof that every user will personally like them.

## 7. Limitations and Future Work

Limitations:

1. The system is based on content similarity, so it does not guarantee user preference.
2. It does not use user history or individual viewing behavior.
3. It does not solve full cold-start and personalization issues.
4. Similarity can be affected by sparse or incomplete overview/keyword metadata.

Future work:

- Use `ratings_small.csv` to build collaborative filtering.
- Combine content similarity and user rating behavior into a hybrid recommender.
- Add user profile controls such as preferred genres or minimum rating.

## 8. Write Recommendation Summary

The markdown summary is saved for README, Streamlit, and presentation use.

In [6]:
summary_lines = [
    "# Movie Recommendation System Summary",
    "",
    "## Objective",
    "This module recommends content-similar movies when a user selects a movie.",
    "It extends the project from analyzing and predicting movie success to helping users discover related movies.",
    "",
    "## Why the IMDB Data Is Suitable",
    "The IMDB metadata contains movie-level content fields such as genres and overview, while keywords.csv provides descriptive keyword tags.",
    "These fields make the dataset suitable for a first content-based recommendation system.",
    "",
    "## Data Fields Used",
    "- movies_metadata.csv: title, genres, overview, vote_average, vote_count, release_date",
    "- keywords.csv: keywords",
    "",
    "## Recommendation Method",
    "For each movie, the system builds a content text using genres, overview, and keywords.",
    "TF-IDF converts the content text into numeric vectors, and cosine similarity measures how similar two movies are.",
    "Movies with higher similarity scores are recommended as more content-similar to the selected movie.",
    "",
    "## Dataset Prepared for Recommendation",
    f"Movies available for recommendation: {recommender_summary['recommendation_rows']:,}.",
    f"Movies with keyword text: {recommender_summary['movies_with_keywords']:,}.",
    f"TF-IDF features: {recommender_summary['tfidf_features']:,}.",
    "",
    "## Sample Recommendation",
    f"The sample recommendation file uses `{sample_title}` as the selected movie and saves the Top 10 results to outputs/sample_recommendations.csv.",
    "",
    "## Limitations",
    "This is based on content similarity and does not mean a user will definitely like the movie.",
    "The current version does not use user history or personalized rating behavior.",
    "It does not fully solve cold-start and individual preference problems.",
    "Some movies have incomplete overviews or keywords, which can weaken similarity quality.",
    "",
    "## Future Work",
    "Use ratings_small.csv for collaborative filtering in a later stage.",
    "Build a hybrid recommender that combines content similarity with user rating behavior.",
]

summary_path = OUTPUT_DIR / "recommendation_summary.md"
summary_path.write_text("\n".join(summary_lines), encoding="utf-8")
print(f"Recommendation summary written to: {summary_path}")

Recommendation summary written to: C:\Users\刘子豪\Desktop\code\IMDB_analysis\outputs\recommendation_summary.md


## 9. Final Checklist

In [7]:
expected_outputs = {
    "recommendation notebook": PROJECT_ROOT / "notebooks" / "04_movie_recommendation.ipynb",
    "recommendation summary": OUTPUT_DIR / "recommendation_summary.md",
    "sample recommendations": OUTPUT_DIR / "sample_recommendations.csv",
}

checklist = pd.DataFrame(
    [(name, str(path.relative_to(PROJECT_ROOT)), path.exists()) for name, path in expected_outputs.items()],
    columns=["Item", "Path", "Exists"],
)
checklist

,Item,Path,Exists
0,recommendation notebook,notebooks\04_movie_recommendation.ipynb,True
1,recommendation summary,outputs\recommendation_summary.md,True
2,sample recommendations,outputs\sample_recommendations.csv,True
